# Review Anomalies

Step through unreviewed quality findings one by one. Inspect the SimFin numbers, open the SEC filing for verification, annotate the reason, mark reviewed.

Reviews are stored in `data/data_quality/anomaly_reviews.toml`. Future runs of `run()` skip already-reviewed (ticker, period, rule) keys.

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

from irp.quality import run, inspect, add_review, load_reviews_df

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 240)

SAMPLE = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'JPM', 'XOM', 'JNJ', 'WMT']
VARIANT = 'A'

## Load Queue

In [2]:
queue = run(SAMPLE, VARIANT, skip_reviewed=True)
print(f'{len(queue)} unreviewed findings')
if len(queue):
    display(queue[['Ticker', 'Period_str', 'Rule', 'rel_diff']].head(20))

37 unreviewed findings


,Ticker,Period_str,Rule,rel_diff
0,AMZN,2024FY,cash_chain,0.133806
1,AMZN,2023FY,cash_chain,0.020522
2,AMZN,2022FY,cash_chain,0.057926
3,AMZN,2021FY,cash_chain,0.061695
4,AMZN,2020FY,cash_chain,0.103570
5,JNJ,2024FY,cash_chain,0.114004
6,JNJ,2023FY,cash_chain,0.014278
7,JNJ,2021FY,cash_chain,0.261765
8,JNJ,2020FY,cash_chain,0.026107
9,META,2024FY,cash_chain,0.231381


## Reviewer

Single-finding loop. `Mark Reviewed & Next` appends to the TOML and advances. `Skip` advances without saving. Re-run `Load Queue` cell to refresh.

In [5]:
state = {'idx': 0}

note_w = widgets.Textarea(
    description='Note:',
    layout=widgets.Layout(width='800px', height='80px'),
)
status_w = widgets.Dropdown(
    options=['ok', 'data_error'],
    description='Status:',
    value='ok',
)
next_btn = widgets.Button(description='Mark Reviewed & Next', button_style='primary')
skip_btn = widgets.Button(description='Skip')
output = widgets.Output()

_NUM_FMT = {c: '{:,.0f}' for c in ['LHS_value', 'RHS_value', 'diff']} | {'rel_diff': '{:.2%}'}


def _fmt_num(x):
    if pd.isna(x):
        return ''
    try:
        return f'{float(x):,.0f}'
    except (TypeError, ValueError):
        return str(x)


def render():
    with output:
        clear_output()
        if state['idx'] >= len(queue):
            print('Queue empty. Re-run the Load Queue cell to refresh.')
            return
        row = queue.iloc[state['idx']]
        progress = f'{state["idx"] + 1}/{len(queue)}'
        link = f'<a href="{row["EDGAR"]}" target="_blank">open EDGAR</a>' if row.get('EDGAR') else '(no EDGAR url)'
        display(HTML(
            f'<h3>[{progress}] {row["Ticker"]} {row["Period_str"]} — {row["Rule"]}</h3>'
            f'<p><b>{row["LHS_value"]:,.0f}</b> vs <b>{row["RHS_value"]:,.0f}</b> '
            f'(diff {row["diff"]:,.0f}, {row["rel_diff"]:.2%}) &nbsp; | &nbsp; {link}</p>'
        ))
        ins = inspect(
            row['Rule'], row['Ticker'],
            int(row['Fiscal Year']), str(row['Fiscal Period']), str(row['Period']),
        )
        if len(ins):
            num_cols = [c for c in ins.columns if ins[c].dtype.kind in 'fi']
            display(ins.style.format({c: _fmt_num for c in num_cols}))
        note_w.value = ''


def on_next(_):
    if state['idx'] >= len(queue):
        return
    row = queue.iloc[state['idx']]
    add_review(row['Ticker'], row['Period_str'], row['Rule'], status_w.value, note_w.value)
    state['idx'] += 1
    render()


def on_skip(_):
    state['idx'] += 1
    render()


next_btn.on_click(on_next)
skip_btn.on_click(on_skip)

display(widgets.VBox([
    output,
    note_w,
    status_w,
    widgets.HBox([next_btn, skip_btn]),
]))
render()

## Audit — All Reviews

In [6]:
_reviews = load_reviews_df()
print(f'{len(_reviews)} reviews recorded')
display(_reviews)

0 reviews recorded


,ticker,period,rule,status,note,reviewed_at
